# Stream A — 03 Entity Scorer

This notebook converts NER + visual grounding outputs into tabular Stream A features.

Input:
- processed manifests
- `cache/stream_a/ner/entities_all.pkl`
- `cache/stream_a/visual_grounding/vg_scores_all.pkl`
- `cache/stream_a/visual_grounding/caption_clip_sim_all.pkl`

Output:
- `cache/stream_a/entity_scorer/features_all.csv`
- dataset-specific feature CSVs


In [1]:
from pathlib import Path
import sys, pickle
import pandas as pd
import numpy as np
from collections import Counter

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import STREAM_A_CACHE, ALL_MANIFEST, FIGURES, TABLES

NER_CACHE = STREAM_A_CACHE / "ner"
VG_CACHE = STREAM_A_CACHE / "visual_grounding"
FEAT_CACHE = STREAM_A_CACHE / "entity_scorer"
FEAT_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("FEAT_CACHE:", FEAT_CACHE)


ROOT: E:\eve-rag
FEAT_CACHE: E:\eve-rag\cache\stream_a\entity_scorer


In [2]:
df_all = pd.read_csv(
    ALL_MANIFEST,
    dtype={"pair_id": "string", "caption": "string", "image_id": "string", "image_path": "string"},
    low_memory=False,
)

ner_all = pickle.load(open(NER_CACHE / "entities_all.pkl", "rb"))
vg_scores = pickle.load(open(VG_CACHE / "vg_scores_all.pkl", "rb"))

caption_sim_path = VG_CACHE / "caption_clip_sim_all.pkl"
caption_sim = pickle.load(open(caption_sim_path, "rb")) if caption_sim_path.exists() else {}

print("Rows:", len(df_all))
print("NER entries:", len(ner_all))
print("VG entries:", len(vg_scores))
print("Caption sim entries:", len(caption_sim))


Rows: 97361
NER entries: 97361
VG entries: 97361
Caption sim entries: 11621


In [3]:
PERSON_TYPES = {"PERSON"}
GPE_TYPES = {"GPE", "LOC", "FAC"}
ORG_TYPES = {"ORG", "NORP"}

def stat_max(vals):
    return float(np.max(vals)) if vals else np.nan

def stat_min(vals):
    return float(np.min(vals)) if vals else np.nan

def stat_mean(vals):
    return float(np.mean(vals)) if vals else np.nan

def make_features(pair_id):
    ents = ner_all.get(pair_id, [])
    scores = vg_scores.get(pair_id, {})

    all_vals, person_vals, gpe_vals, org_vals = [], [], [], []
    ent_types = Counter()

    for e in ents:
        text = str(e.get("text", "")).strip()
        label = str(e.get("label", ""))
        ent_types[label] += 1
        if text in scores:
            val = scores[text]
            all_vals.append(val)
            if label in PERSON_TYPES:
                person_vals.append(val)
            elif label in GPE_TYPES:
                gpe_vals.append(val)
            elif label in ORG_TYPES:
                org_vals.append(val)

    return {
        "n_entities": len(ents),
        "n_person": ent_types.get("PERSON", 0),
        "n_gpe": ent_types.get("GPE", 0) + ent_types.get("LOC", 0) + ent_types.get("FAC", 0),
        "n_org": ent_types.get("ORG", 0) + ent_types.get("NORP", 0),
        "max_grounding": stat_max(all_vals),
        "min_grounding": stat_min(all_vals),
        "mean_grounding": stat_mean(all_vals),
        "person_max": stat_max(person_vals),
        "gpe_max": stat_max(gpe_vals),
        "org_max": stat_max(org_vals),
        "caption_clip_sim": caption_sim.get(pair_id, np.nan),
        "n_grounded_entities": len(all_vals),
    }


In [4]:
feature_rows = []

for row in df_all.itertuples(index=False):
    pair_id = str(row.pair_id)
    feats = make_features(pair_id)
    feature_rows.append({
        "pair_id": pair_id,
        "dataset": row.dataset,
        "split": row.split,
        "label": int(row.label) if not pd.isna(row.label) else np.nan,
        "has_image": int(row.has_image),
        **feats,
    })

features = pd.DataFrame(feature_rows)
features.to_csv(FEAT_CACHE / "features_all.csv", index=False)
pickle.dump(features, open(FEAT_CACHE / "features_all.pkl", "wb"), protocol=4)

for (dataset, split), sub in features.groupby(["dataset", "split"]):
    name = f"features_{dataset}_{split}.csv"
    sub.to_csv(FEAT_CACHE / name, index=False)

for dataset, sub in features.groupby("dataset"):
    sub.to_csv(FEAT_CACHE / f"features_{dataset}.csv", index=False)

print("Saved features:", features.shape)
features.head()


Saved features: (97361, 17)


,pair_id,dataset,split,label,has_image,n_entities,n_person,n_gpe,n_org,max_grounding,min_grounding,mean_grounding,person_max,gpe_max,org_max,caption_clip_sim,n_grounded_entities
0,nc_620701_0,newsclippings,train,0,0,3,0,2,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,nc_620701_1,newsclippings,train,1,0,3,0,2,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,nc_1569820_0,newsclippings,train,0,0,2,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,nc_1569820_1,newsclippings,train,1,0,3,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,nc_1490080_0,newsclippings,train,0,0,1,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [5]:
summary = []
for dataset, sub in features.groupby("dataset"):
    summary.append({
        "dataset": dataset,
        "rows": len(sub),
        "has_image": int(sub["has_image"].sum()),
        "mean_entities": float(sub["n_entities"].mean()),
        "rows_with_grounding": int((sub["n_grounded_entities"] > 0).sum()),
        "caption_clip_sim_available": int(sub["caption_clip_sim"].notna().sum()),
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(TABLES / "entity_scorer_feature_summary.csv", index=False)
summary_df


,dataset,rows,has_image,mean_entities,rows_with_grounding,caption_clip_sim_available
0,mmfakebench,11000,11000,1.629727,7661,11000
1,newsclippings,85360,0,3.035918,0,0
2,verite,1001,621,2.925075,584,621


In [6]:
print("=" * 60)
print("Entity scorer complete.")
print("Feature files:")
for p in sorted(FEAT_CACHE.glob("*")):
    print(" ", p.relative_to(ROOT), f"{p.stat().st_size/1e6:.2f} MB")
print("Next: stream_a_results_v2_updated.ipynb")
print("=" * 60)


Entity scorer complete.
Feature files:
  cache\stream_a\entity_scorer\features_all.csv 6.24 MB
  cache\stream_a\entity_scorer\features_all.pkl 16.14 MB
  cache\stream_a\entity_scorer\features_mmfake.pkl 0.99 MB
  cache\stream_a\entity_scorer\features_mmfakebench.csv 1.53 MB
  cache\stream_a\entity_scorer\features_mmfakebench_test.csv 1.39 MB
  cache\stream_a\entity_scorer\features_mmfakebench_val.csv 0.14 MB
  cache\stream_a\entity_scorer\features_newsclippings.csv 4.61 MB
  cache\stream_a\entity_scorer\features_newsclippings_test.csv 0.39 MB
  cache\stream_a\entity_scorer\features_newsclippings_test.pkl 0.70 MB
  cache\stream_a\entity_scorer\features_newsclippings_train.csv 3.85 MB
  cache\stream_a\entity_scorer\features_newsclippings_train.pkl 6.84 MB
  cache\stream_a\entity_scorer\features_newsclippings_val.csv 0.37 MB
  cache\stream_a\entity_scorer\features_newsclippings_val.pkl 0.68 MB
  cache\stream_a\entity_scorer\features_verite.csv 0.11 MB
  cache\stream_a\entity_scorer\featur